# KALIA X18 — close F-3 and F-4

X18 ran three arms (control / NoPE / gated) and the GPU kernel measured only
validation loss. Two registered predictions were therefore left unadjudicated:

- **F-3** — *gated improves accuracy by more than it improves loss*: needs the
  accuracy half, which was never run.
- **F-4** — *the gate actually opens (mean > 0.05)*: needs a gate diagnostic,
  which the kernel did not emit.

F-1 and F-2 are already settled from validation loss alone and need nothing
here. This kernel supplies the two missing measurements. CPU, so it costs no
GPU quota and runs alongside the v0.2.0 session.

In [ ]:
# Same pin as the v0.2.0 bench kernel: lm-eval 0.4.9 imports
# AutoModelForVision2Seq, removed in transformers v5.
!pip install -q "lm-eval==0.4.9" "transformers<5" tiktoken pyyaml datasets
import glob, importlib.metadata as md, shutil, sys
for pkg in ('lm_eval', 'transformers', 'torch', 'datasets'):
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except Exception as exc:
        print(f"{pkg:14s} MISSING ({exc})")

In [ ]:
root = sorted(glob.glob("/kaggle/input/datasets/subhajitlucky/kalia-code-dev"))[0]
import os
for f in ["eval_bench.py", "kalia_lm.py", "model.py", "data.py",
          "eval_reversibility.py", "gate_probe.py"]:
    shutil.copy(f"{root}/{f}", "/kaggle/working/")
    print("copied", f)
# eval/ too: gate_probe.py resolves its probe sentences from
# eval/probe_sentences.json, and the dataset used to be flat with no JSON at
# all. Omitting this is what killed version 1 of this kernel, and the same
# omission is why X18's Abhimanyu gap was never measurable.
os.makedirs("/kaggle/working/eval", exist_ok=True)
for f in glob.glob(f"{root}/eval/*.json"):
    shutil.copy(f, "/kaggle/working/eval/")
    print("copied eval/" + os.path.basename(f))
assert os.path.exists("/kaggle/working/eval/probe_sentences.json"), (
    "probe_sentences.json missing -- is the kalia-code-dev dataset current?")
os.chdir("/kaggle/working")
ckpts = sorted(glob.glob(
    "/kaggle/input/notebooks/subhajitlucky/kalia-x18-ablate/out/*/ckpt.pt"))
print("checkpoints:", *ckpts, sep="\n  ")
assert len(ckpts) == 3, f"expected 3 arms, found {len(ckpts)}"

## F-4 first — it decides how F-3 should be read

If the gate is inert, then any accuracy difference on the gated arm belongs
to the branch-normalisation structure rather than to the data-dependent read
that Gated Residual actually claims, and F-3's question changes shape. So
measure the gate before spending CPU on the benchmark.

In [ ]:
# Secondary, informational: the 20 fixed probe sentences, for comparison with
# the locally measured 0.0192. Deliberately non-fatal -- the primary F-4
# measurement (canonical val batches) follows, and an optional view must not
# be able to take the kernel down with it.
import subprocess
gated = [c for c in ckpts if 'micro-gated' in c][0]
print('gated checkpoint:', gated)
rc = subprocess.run(
    [sys.executable, 'gate_probe.py',
     '--ckpt', gated,
     '--sentences', 'eval/probe_sentences.json',
     '--json-out', '/kaggle/working/gate_f4_sentences.json'],
).returncode
print('probe-sentence F-4 exit code:', rc, '(0 = ok)')

In [ ]:
# F-4 on the canonical validation set, not the 20 short probe sentences.
# The prediction says "mean gate value at the end"; the end of training is
# best estimated on real val text in full-length windows.
import json
import numpy as np
import torch
import gate_probe as gp
from eval_reversibility import load_model

val_bin = sorted(glob.glob(
    "/kaggle/input/notebooks/subhajitlucky/kalia-prep-v2b/**/val.bin",
    recursive=True))
assert val_bin, "attach the kalia-prep-v2b notebook output for val.bin"
print('val.bin:', val_bin[0])

device = torch.device('cpu')
model = load_model(gated, device)
ctx = model.cfg.context_len
print('context_len:', ctx)

arr = np.memmap(val_bin[0], dtype=np.uint16, mode='r')
print('val tokens:', len(arr))
batches = []
for i in range(100):
    start = i * ctx
    if start + ctx > len(arr):
        break
    ids = torch.tensor(arr[start:start + ctx], dtype=torch.long).unsqueeze(0)
    batches.append(ids)
print('batches:', len(batches), 'tokens each:', ctx)
res = gp.gate_means(model, batches)
res['batches'] = len(batches)
res['init_mean'] = gp.INIT_GATE_MEAN
res['delta'] = res['overall'] - gp.INIT_GATE_MEAN
res['f4_threshold'] = gp.F4_THRESHOLD
res['f4_passes'] = bool(res['overall'] > gp.F4_THRESHOLD)
for i, v in enumerate(res['per_block']):
    print(f'  block {i}: mean gate {v:.4f}')
print(f"  overall {res['overall']:.4f} | init {res['init_mean']:.4f} "
      f"| delta {res['delta']:+.4f} | threshold {res['f4_threshold']} "
      f"| F-4 {'PASS' if res['f4_passes'] else 'FAIL'}")
with open('/kaggle/working/gate_f4_val.json', 'w') as fh:
    json.dump(res, fh, indent=2)

## F-3 — does the accuracy gain exceed the loss gain?

Registered threshold: gated must be at least control on val loss (it is, by
0.0436) **and** at least 0.5 pp better on a downstream task. Same five tasks,
0-shot, 500 samples, as every other number in this project.

In [ ]:
import json
import subprocess
for arm in ('micro-base', 'micro-gated'):
    ck = [c for c in ckpts if arm in c][0]
    print('=' * 64)
    print('ARM:', arm)
    subprocess.run(
        [sys.executable, 'eval_bench.py',
         '--ckpt', ck,
         '--out', f'/kaggle/working/bench_{arm}.json',
         '--limit', '500'],
        check=True,
    )

In [ ]:
base = json.load(open('/kaggle/working/bench_micro-base.json'))
gated = json.load(open('/kaggle/working/bench_micro-gated.json'))

def acc(res, task):
    m = res[task]
    for key in ('acc_norm,none', 'acc,none'):
        if key in m:
            return m[key] * 100
    raise KeyError(task)

tasks = sorted(set(base) & set(gated))
print('F-3: gated minus control, in accuracy points')
print(f"{'task':24s} {'control':>9s} {'gated':>9s} {'delta':>8s}")
deltas = []
for t in tasks:
    b, g = acc(base, t), acc(gated, t)
    deltas.append(g - b)
    flag = '  <- >=0.5pp' if (g - b) >= 0.5 else ''
    print(f'{t:24s} {b:9.1f} {g:9.1f} {g-b:+8.2f}{flag}')
best = max(deltas)
print(f"\nbest task delta {best:+.2f} pp | F-3 needs >= 0.5 pp "
      f"-> {'PASS' if best >= 0.5 else 'FAIL'}")
print('val loss delta was -0.0436 nats (gated better)')

Read the two verdicts together, not separately.

- **F-4 fail + F-3 fail** → the gate is inert *and* buys no accuracy, so the
  −0.0436 nats belongs to the branch normalisation. Keep the normalisation,
  drop the gate, and register the isolation arm (X19).
- **F-4 fail + F-3 pass** → the accuracy gain is real but is not coming from
  the gate, which is a more interesting and more awkward result: a claim
  borrowed from Qwen's 125B does not survive being reimplemented at 30M.
- **F-4 pass** → the pre-registered F-3 question is answerable as written, and
  the loss/accuracy divergence Qwen reported is reproduced at our scale.

All outcomes are reported. The pre-registration's own caveat applies: a null at
30M over 500 steps is weak evidence about 125B, and this is a screening run, not
a training run.